# 🎨 AI Illustration Images (no real photos) — One-click Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/ai_images_colab.ipynb)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Task](https://img.shields.io/badge/task-seg-4c8cbf) ![Auto](https://img.shields.io/badge/runs-automatically-3fb950)

Generates **AI-style satellite illustrations** — stylized, fictional disaster scenes for slides, READMEs and posts, with an explicit `AI-GENERATED · NOT A REAL PHOTO` watermark. Default engine is a fast procedural style-transfer stack (posterize + palette + grain + vignette); an optional cell adds true Stable-Diffusion images when a GPU runtime is on. Optionally upload a base image to stylize your own scene.

| ▶️ How to run | |
|---|---|
| **Everything automatic** | Click **Runtime ▸ Run all** (Ctrl+F9): setup → demo scene → inference → dashboard → export chain without any further clicks |
| **With your data** | Run the cells once, then click the 📤 upload buttons — the pipeline re-runs automatically after upload |
| **No image at hand?** | The 🛟 fallback generates a clearly-labeled synthetic flood scene — the notebook *never blocks or crashes* |

**📤 Upload preview — what to have ready:**

| file | what it is | typical size |
|---|---|---|
| **`base_scene.png`** | OPTIONAL — image to stylize (else synthetic scenes) | 512–2048 px |
| **`no file?`** | click 🛟 — synthetic flood scene is generated | — |

> 🔖 Every result is stamped with its mode: **CHECKPOINT** (your `.pth`), **QUICK-TRAIN** (genuinely briefly-trained), or **SYNTHETIC-DEMO** (clearly-labeled heuristic). Nothing is faked.

In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent, restart-safe {display-mode:"form"}
AUTO_RUN = True  #@param {type:"boolean"}
SHOW_PREVIEW_ON_UPLOAD = True  #@param {type:"boolean"}
import os, sys, glob, json, math, random, subprocess, traceback, datetime, textwrap

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

colab_kit.pip_install("ipywidgets", "pillow", "numpy", "imageio", "scikit-image", "tifffile", "matplotlib")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageDraw
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
from colab_kit import (make_synthetic, infer, overlay, heatmap, to_u8, stats_for,
                       stylize_poster, read_any, verdict, show_pair, run_inference,
                       run_dashboard, export_results_zip_a, nb_save_upload, nb_apply,
                       nb_upload_pre, nb_upload_post, nb_use_synthetic,
                       _field, _fbm, _stretch, _blur, _median_mask, _terrain, _hsv,
                       segment_heuristic, change_heuristic, _otsu)

random.seed(42); np.random.seed(42)
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
DOMAIN = "flood"          # this notebook's disaster domain
TASK   = "seg"            # "seg" | "cd" | "both"
GSD_M  = 10.0               # ground sample distance (m/pixel) for km² stats
NOTEBOOK = "ai_images_colab.ipynb"
STATE = {"domain": DOMAIN, "task": "seg" if TASK != "cd" else "cd",
         "pre": None, "post": None, "gt": None, "mask": None, "source": "none"}
print("✅ Environment ready — notebook:", NOTEBOOK, "| domain:", DOMAIN, "| task:", TASK)

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — buttons + inline preview, never blocks {display-mode:"form"}
# upload logic lives in colab_kit (single source of truth, unit-tested in CI)
def _save_upload(name, data):
    return colab_kit.nb_save_upload(globals(), name, data)

def _apply(kind, arr):
    return colab_kit.nb_apply(globals(), kind, arr)

def upload_pre(_=None):
    return colab_kit.nb_upload_pre(globals())

def upload_post(_=None):
    return colab_kit.nb_upload_post(globals())

def use_synthetic(_=None):
    return colab_kit.nb_use_synthetic(globals())


SPEC = [['base_scene.png',
  'OPTIONAL — image to stylize (else synthetic scenes)',
  '512–2048 px'],
 ['no file?', 'click 🛟 — synthetic flood scene is generated', '—']]
W = 70
def print_spec():
    print("┌" + "─" * W + "┐")
    for line in textwrap.wrap("📤 WHAT TO UPLOAD — " + NOTEBOOK, W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    for label, desc, size in SPEC:
        wrapped = textwrap.wrap(label + " — " + desc + "  (" + size + ")", W - 6)
        for j, line in enumerate(wrapped):
            print(("│   • " if j == 0 else "│     ") + line.ljust(W - 6) + "│")
    for line in textwrap.wrap(
            "accepted: .png .jpg .jpeg .tif .tiff .h5 | sweet spot: 512–2048 px, ≤ 20 MB each | "
            "GeoTIFF georeference is kept when present", W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    print("└" + "─" * W + "┘")
print_spec()

out_up = widgets.Output(layout={"border": "1px solid #ddd"})
domain_sel = widgets.Dropdown(
    options=["flood", "landslide", "wildfire", "oil_spill", "avalanche",
             "volcano", "glof", "dust_storm", "smoke_plume", "locust",
             "tsunami", "hurricane", "infra_damage"],
    value="flood", description="Domain:")
def _sync_dom(*_):
    STATE["domain"] = domain_sel.value
domain_sel.observe(_sync_dom, names="value")
# (task fixed for this notebook)
gsd_box = widgets.FloatText(value=GSD_M, description="GSD m/px:",
                            tooltip="ground sample distance, used for km² statistics")
# (single-image task — no POST button needed)
btn_extra = None
# (no extra upload buttons)

btn_pre = widgets.Button(description="⬆️ Upload PRE image", button_style="primary", icon="upload")
btn_synth = widgets.Button(description="🛟 Use bundled synthetic sample",
                           button_style="warning", icon="picture")
btns = [btn_pre]
if TASK in ("cd", "both"):
    btns.insert(1, btn_post)
if btn_extra is not None:
    btns.append(btn_extra)
box = widgets.VBox([widgets.HBox(btns), widgets.HBox([btn_synth, gsd_box]), out_up])






btn_pre.on_click(upload_pre)
if TASK in ("cd", "both"):
    btn_post.on_click(upload_post)
btn_synth.on_click(use_synthetic)
display(box)

if AUTO_RUN and STATE["pre"] is None:
    print("⚡ AUTO_RUN: generating bundled synthetic sample — upload later to re-run automatically")
    use_synthetic()

In [ ]:
#@title 🎨 GENERATE — AI-style illustration gallery (no real photos) {display-mode:"form"}
N_IMAGES = 6  #@param {type:"integer"}
POSTER_STYLE = True  #@param {type:"boolean"}
MODE_TAG = "AI-ILLUSTRATION"
from PIL import ImageDraw
os.makedirs(os.path.join(OUT_DIR, "ai"), exist_ok=True)

def _watermark(img_u8, text):
    im = Image.fromarray(img_u8)
    d = ImageDraw.Draw(im)
    bar = 26
    d.rectangle([0, im.height - bar, im.width, im.height], fill=(12, 12, 14))
    d.text((10, im.height - bar + 7), text, fill=(255, 255, 255))
    return np.asarray(im)

def run_ai_gallery():
    STATE["mode_tag"] = MODE_TAG
    domain = STATE["domain"]
    base = STATE.get("post") or STATE.get("pre")
    paths = []
    for k in range(max(1, min(12, N_IMAGES))):
        seed = random.randint(0, 99999)
        if base is not None and k == 0 and STATE.get("source") == "user":
            scene = base
        else:
            _, scene, _ = make_synthetic(domain, 512, 384, seed=seed)
        art = stylize_poster(scene, k=5 + (k % 4)) if POSTER_STYLE else scene
        u8 = _watermark(to_u8(art),
                        "AI-GENERATED ILLUSTRATION | " + domain + " | seed " + str(seed) +
                        " | NOT A REAL PHOTO")
        p = os.path.join(OUT_DIR, "ai", "ai_" + domain + "_" + str(seed) + ".png")
        Image.fromarray(u8).save(p)
        paths.append(p)
    cols = 3
    rows = math.ceil(len(paths) / cols)
    fig, ax = plt.subplots(rows, cols, figsize=(4.4 * cols, 3.4 * rows))
    flat = np.atleast_1d(ax).ravel()
    for a, p in zip(flat, paths):
        a.imshow(plt.imread(p)); a.set_title(os.path.basename(p), fontsize=8); a.axis("off")
    for a in flat[len(paths):]:
        a.axis("off")
    plt.suptitle("AI-style illustrations — procedurally generated fictional scenes")
    plt.show()
    print("🖼️ saved", len(paths), "illustrations →", os.path.join(OUT_DIR, "ai"))
    if AUTO_RUN and "run_export" in globals():
        run_export()

run_ai_gallery() if AUTO_RUN else print("call run_ai_gallery() when ready")

In [ ]:
#@title 🎇 OPTIONAL — true diffusion images (needs GPU runtime, ~4 GB download) {display-mode:"form"}
USE_DIFFUSERS = False  #@param {type:"boolean"}
DIFF_PROMPT = "aerial satellite view of a flooded town, stylized illustration, vivid colors"  #@param {type:"string"}
if USE_DIFFUSERS:
    try:
        import torch
        from diffusers import StableDiffusionPipeline
        pipe = StableDiffusionPipeline.from_pretrained(
            "stabilityai/sd-turbo", torch_dtype=torch.float16).to("cuda")
        os.makedirs(os.path.join(OUT_DIR, "ai_diffusion"), exist_ok=True)
        for k in range(4):
            img = pipe(DIFF_PROMPT, num_inference_steps=30, guidance_scale=7.0).images[0]
            p = os.path.join(OUT_DIR, "ai_diffusion", "diff_" + str(k) + ".png")
            img.save(p)
            print("saved", p)
    except Exception as e:
        print("Diffusion skipped —", e)
        print("💡 needs GPU runtime (Runtime ▸ Change runtime type ▸ T4 GPU) + first-run download")
else:
    print("skipped (USE_DIFFUSERS=False) — the procedural gallery above already covers "
          "illustration mode with zero downloads")

In [ ]:
#@title 📦 EXPORT — results zip + optional Drive copy + auto-download {{display-mode:"form"}}
USE_DRIVE = False  #@param {{type:"boolean"}}

def run_export():
    return colab_kit.export_results_zip_a(globals(), use_drive=USE_DRIVE)

run_export() if AUTO_RUN else print("call run_export() when ready")

## 🖼️ About these images

- Every output is **procedurally generated or diffusion-generated** — no real photos, no licensing risk.
- Each PNG carries a visible **AI-GENERATED · NOT A REAL PHOTO** watermark; keep it when sharing.
- `assets/CREDITS.md` in the repo documents the same policy for README artwork.
- Want a GIF for your README? Open **`demo_gif_colab.ipynb`** next.